## Case When formatting test

### Goals

Answer question from `orbital` team: 

> Does "unnesting" queries improve performance for trees? If only sometimes, where/when?

Research design:

- Simulate SQL code for medium-size random forest:
  + Max Depth: 4 (2**4 terminal nodes)
  + Trees: 100
- Test three different varieties for performance:
  + Original nested CASE WHEN
  + "Linearized" CASE WHEN (pulling all conditions to top level)
  + "Smart" Linearized -- same as above but with refundant conditions removed
- Test on two different data distributions:
  + Uniformly distributed (all nodes have same chance of terminating)
  + Skewed distribution (to simulate if we re-rodered query to take advantage of CASE WHEN early term)


### Set Up -- Tree Code

In [1]:
import duckdb
import polars as pl
import polars.selectors as cs
import sqlglot
import pandas as pd
import numpy as np

In [2]:
# nested case when version
d = []
for i in np.arange(8):
    d += [ f"case when d{i+1} > 0.5 then {i+1} else 0 end"]

c = []
for i in np.arange(4):
    sql_temp = f"case when c{i+1} > 0.5 then {d[2*i]} else {d[2*i+1]} end"
    c += [sql_temp]

b = []
for i in np.arange(2):
    sql_temp = f"case when b{i+1} > 0.5 then {c[2*i]} else {c[2*i+1]} end"
    b += [sql_temp]

sql_nest = f"case when a > 0.5 then {b[0]} else {b[1]} end"

In [3]:
# linearize case when version 
# using strict versus equalities versus inclusion / exclusion just so my code lines up nice =) 
# it's a timing exercise so it really doesn't matter
sql_line = '''
case
-- a > 0.5
when a > 0.5 and b1 > 0.5 and c1 > 0.5 and d1 > 0.5 then 1
when a > 0.5 and b1 > 0.5 and c1 > 0.5 and d1 < 0.5 then 0 
when a > 0.5 and b1 > 0.5 and c1 < 0.5 and d2 > 0.5 then 2
when a > 0.5 and b1 > 0.5 and c1 < 0.5 and d2 < 0.5 then 0
when a > 0.5 and b1 < 0.5 and c2 > 0.5 and d3 > 0.5 then 3
when a > 0.5 and b1 < 0.5 and c2 > 0.5 and d3 < 0.5 then 0 
when a > 0.5 and b1 < 0.5 and c2 < 0.5 and d4 > 0.5 then 4
when a > 0.5 and b1 < 0.5 and c2 < 0.5 and d4 < 0.5 then 0
-- a <= 0.5
when a < 0.5 and b2 > 0.5 and c3 > 0.5 and d5 > 0.5 then 5
when a < 0.5 and b2 > 0.5 and c3 > 0.5 and d5 < 0.5 then 0 
when a < 0.5 and b2 > 0.5 and c3 < 0.5 and d6 > 0.5 then 6
when a < 0.5 and b2 > 0.5 and c3 < 0.5 and d6 < 0.5 then 0
when a < 0.5 and b2 < 0.5 and c4 > 0.5 and d7 > 0.5 then 7
when a < 0.5 and b2 < 0.5 and c4 > 0.5 and d7 < 0.5 then 0 
when a < 0.5 and b2 < 0.5 and c4 < 0.5 and d8 > 0.5 then 8
when a < 0.5 and b2 < 0.5 and c4 < 0.5 and d8 < 0.5 then 0
else null end
'''

In [4]:
# linearized case when version -- pruning redundant conditions
sql_slim = '''
case

when a > 0.5 and b1 > 0.5 and c1 > 0.5 and d1 > 0.5 then 1
when a > 0.5 and b1 > 0.5 and c1 > 0.5              then 0 
when a > 0.5 and b1 > 0.5              and d2 > 0.5 then 2
when a > 0.5 and b1 > 0.5                           then 0
when a > 0.5 and              c2 > 0.5 and d3 > 0.5 then 3
when a > 0.5 and              c2 > 0.5              then 0 
when a > 0.5 and                           d4 > 0.5 then 4
when a > 0.5                                        then 0

when             b2 > 0.5 and c3 > 0.5 and d5 > 0.5 then 5
when             b2 > 0.5 and c3 > 0.5              then 0 
when             b2 > 0.5              and d6 > 0.5 then 6
when             b2 > 0.5                           then 0
when                          c4 > 0.5 and d7 > 0.5 then 7
when                          c4 > 0.5              then 0 
when                                       d8 > 0.5 then 8
when a < 0.5                                        then 0
else null end
'''

### Set Up -- Data

In [5]:
# set up random data matrix
n = 1000000
p = 15
df = pl.DataFrame( np.random.rand(n,p) )
df.columns = ['a','b1','b2','c1','c2','c3','c4','d1','d2','d3','d4','d5','d6','d7','d8']
df.glimpse()

Rows: 1000000
Columns: 15
$ a  <f64> 0.7242264084959612, 0.3507249932944493, 0.7173525361865358, 0.810404291595001, 0.7254552195154176, 0.17774042712805194, 0.6666191854678684, 0.7330507331141142, 0.9929884037966842, 0.8458396736759629
$ b1 <f64> 0.5711345024949848, 0.5249102319955556, 0.2566103338823603, 0.9753374655717109, 0.2683392519934772, 0.7024973431577116, 0.8040901264398025, 0.3250194829385905, 0.4834264820111027, 0.5490951341562894
$ b2 <f64> 0.7898404338820595, 0.4846382732982615, 0.9402935033489761, 0.6481104915048125, 0.03173737103780927, 0.4739492007989935, 0.07024954352037327, 0.9708288745909244, 0.2603196397418509, 0.7694715732254096
$ c1 <f64> 0.4232111244084583, 0.07541724509868608, 0.6391045287360344, 0.5424258434826041, 0.247302137530079, 0.8228433250627295, 0.26338177455477185, 0.8409816865643482, 0.4438166025428226, 0.22590436371815192
$ c2 <f64> 0.4160585239036446, 0.2145809845949609, 0.12615300744907354, 0.03141907383983078, 0.6447455033323718, 0.656612831151984

In [6]:
# ensure different candidates have same logic
sql_compare = f"""
select
{sql_nest} as out_nest,
{sql_line} as out_line,
{sql_slim} as out_slim,
*
from df
"""
df_out = duckdb.sql(sql_compare).pl()
df_out.filter( 
    (pl.col('out_line') != pl.col('out_slim')) |
    (pl.col('out_nest') != pl.col('out_slim'))
)

out_nest,out_line,out_slim,a,b1,b2,c1,c2,c3,c4,d1,d2,d3,d4,d5,d6,d7,d8
i32,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64


In [7]:
# base case output frequency
df_out.group_by('out_nest').len()

out_nest,len
i32,u32
2,62205
4,62229
8,62411
7,62994
1,62065
5,62496
6,62739
3,62802
0,500059


In [8]:
# create different skews
df_skew = (
df_out
.with_columns(threshhold = pl.when(pl.col('out_nest') > 0).then(10 - pl.col('out_nest')).otherwise(5))
.filter( pl.col('out_nest').cum_count().over('out_nest') <= pl.col('threshhold')*7000 )
)

print(df_skew.shape[0])

(
df_skew
.group_by('out_nest')
.len()
.sort('out_nest')
.with_columns( p = pl.col('len') / pl.col('len').sum() )
)

342065
342065


out_nest,len,p
i32,u32,f64
0,35000,0.10232
1,62065,0.181442
2,56000,0.163712
3,49000,0.143248
4,42000,0.122784
5,35000,0.10232
6,28000,0.081856
7,21000,0.061392
8,14000,0.040928


In [9]:
# final prep - standardizing data size
df_skew = pl.concat([df_skew]*4).drop( cs.starts_with('out_') )
df_unif = pl.DataFrame( np.random.rand( df_skew.shape[0],p) )
df_unif.columns = ['a','b1','b2','c1','c2','c3','c4','d1','d2','d3','d4','d5','d6','d7','d8']

### Timing

In [10]:
con = duckdb.connect()
con.sql("SET enable_object_cache = false;")

#### Base Case

In this case all nodes are equally likely

In [11]:
qry_nest = f"select {'+'.join([sql_nest]*100)} as pred from df_unif"
qry_line = f"select {'+'.join([sql_line]*100)} as pred from df_unif"
qry_slim = f"select {'+'.join([sql_slim]*100)} as pred from df_unif"
qry_ctes = f"""
    with trees as (
    select { ','.join([f"{sql_slim} as t{i}" for i in np.arange(100)]) } 
    from df_unif
    ) 
    select {'+'.join([f"t{i}" for i in np.arange(100)])} as pred 
    from trees
    """

In [12]:
%%timeit -n 1 -r 50

con.sql(qry_nest).execute()

485 ms ± 63 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
485 ms ± 63 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [13]:
%%timeit -n 1 -r 50

con.sql(qry_line).execute()

1.94 s ± 361 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
1.94 s ± 361 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [14]:
%%timeit -n 1 -r 50

con.sql(qry_slim).execute()

987 ms ± 52.2 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
987 ms ± 52.2 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [15]:
%%timeit -n 1 -r 50

con.sql(qry_ctes).execute()

256 ms ± 9.75 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
256 ms ± 9.75 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


#### Skewed Case

In this case, nodes are skewed, so we can see benefit from the linearized version ordering by node size 

In [16]:
qry_nest = f"select {'+'.join([sql_nest]*100)} as pred from df_skew"
qry_line = f"select {'+'.join([sql_line]*100)} as pred from df_skew"
qry_slim = f"select {'+'.join([sql_slim]*100)} as pred from df_skew"

In [17]:
%%timeit -n 1 -r 50

con.sql(qry_nest).execute()

479 ms ± 74.8 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
479 ms ± 74.8 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [18]:
%%timeit -n 1 -r 50

con.sql(qry_line).execute()

1.91 s ± 650 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
1.91 s ± 650 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [19]:
%%timeit -n 1 -r 50

con.sql(qry_slim).execute()

965 ms ± 23.1 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
965 ms ± 23.1 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [20]:
%%timeit -n 1 -r 50

con.sql(qry_ctes).execute()

260 ms ± 8.25 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
260 ms ± 8.25 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


#### Sizes

In [21]:
def kb(s): return len(s.encode('utf-8')) / 1e3

In [22]:
qry_titl = ['Nest', 'Line', 'Slim', 'CTEs']
qry_lean = [
sqlglot.transpile(qry_nest, write='duckdb', identify=False, pretty=False, indent=0, pad=0)[0],
sqlglot.transpile(qry_line, write='duckdb', identify=False, pretty=False, indent=0, pad=0)[0],
sqlglot.transpile(qry_slim, write='duckdb', identify=False, pretty=False, indent=0, pad=0)[0],
sqlglot.transpile(qry_ctes, write='duckdb', identify=False, pretty=False, indent=0, pad=0)[0]
]
qry_size = [kb(q) for q in qry_lean]

pl.DataFrame({
    'Query': qry_titl,
    'Size (KB)': qry_size
}).sort('Size (KB)')

Query,Size (KB)
str,f64
"""Nest""",52.825
"""Slim""",56.925
"""CTEs""",58.138
"""Line""",96.525
